# Tsuki epoch-0 Base/full-ICL Colab TTS service

This notebook serves the pinned public `epoch-0` checkpoint from Hugging Face on a Colab GPU. It uses the authorized `tsuki_25s_clip.mp3` and `tsuki_ref_text.txt` as a full-ICL voice-clone prompt and exposes an authenticated `/health` and `/tts` API for the VPS.

The service is explicitly `trained-base-full-icl`; it is not the separate `trained-custom` speaker-ID service and does not use `generate_custom_voice`. Select a T4 GPU before running. The Hugging Face repository is public, so no HF token is required.

In [ ]:
%pip install -q qwen-tts huggingface_hub fastapi uvicorn soundfile requests

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import time

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

HF_REPO_ID = 'tsukihime-exe/tsuki-qwen3-tts-models'
HF_REVISION = '044462bd5ca834ba7ac44ff044d125decda6e863'
MODEL_ROOT = Path(snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type='model',
    revision=HF_REVISION,
    allow_patterns=['epoch-0/*'],
    local_dir='/content/tsuki_epoch0_public',
))
MODEL_PATH = MODEL_ROOT / 'epoch-0'
REFERENCE_ROOT = Path('/content/tsuki_epoch0_public/reference')
REFERENCE_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_AUDIO = Path(hf_hub_download(
    repo_id=HF_REPO_ID, filename='tsuki_25s_clip.mp3',
    repo_type='model', revision=HF_REVISION, local_dir=REFERENCE_ROOT,
))
REFERENCE_TEXT = Path(hf_hub_download(
    repo_id=HF_REPO_ID, filename='tsuki_ref_text.txt',
    repo_type='model', revision=HF_REVISION, local_dir=REFERENCE_ROOT,
))

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert not HfApi().model_info(HF_REPO_ID, revision=HF_REVISION).private
assert sha256_file(REFERENCE_AUDIO) == 'ce93bb53f197f0683d46ab4ad240e3d434ef6e2b2340190be98dfe382a0f6530'
assert sha256_file(REFERENCE_TEXT) == 'bd3e849d943b80e75f62243c85c3d75dabe2716c2d1b12535b39960d9406949e'
reference_text = REFERENCE_TEXT.read_text(encoding='utf-8').strip()
print('Public checkpoint and reference files verified:', MODEL_PATH)

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required. Select a Colab T4 runtime.')
model = Qwen3TTSModel.from_pretrained(
    MODEL_PATH,
    device_map='cuda:0',
    dtype=torch.bfloat16,
    attn_implementation='eager',
)
voice_clone_prompt = model.create_voice_clone_prompt(
    ref_audio=str(REFERENCE_AUDIO),
    ref_text=reference_text,
    x_vector_only_mode=False,
)
print('Loaded trained Base/full-ICL epoch-0 on', torch.cuda.get_device_name(0))

In [ ]:
import asyncio
import getpass
import hmac
import io
import threading

import numpy as np
import soundfile as sf
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel, Field

_api_key_file = Path('/content/tsuki_colab_tts_api_key')
if _api_key_file.is_file():
    os.environ['TSUKI_COLAB_TTS_API_KEY'] = _api_key_file.read_text(encoding='utf-8').strip()
    _api_key_file.unlink()
API_KEY = os.environ.get('TSUKI_COLAB_TTS_API_KEY') or getpass.getpass('Colab TTS API key: ')
if not API_KEY:
    raise RuntimeError('TSUKI_COLAB_TTS_API_KEY is required')
MAX_TEXT_CHARS = 280
MAX_NEW_TOKENS = 512
MODEL_ID = f'{HF_REPO_ID}/epoch-0'
BASE_MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'
GENERATION_VERSION = 'hf-epoch0-base-full-icl-v1'
MODEL_LOCK = asyncio.Lock()
app = FastAPI(title='Tsuki epoch-0 Base/full-ICL Colab TTS', version=GENERATION_VERSION)

def _authorize(value):
    if not value or not hmac.compare_digest(value, API_KEY):
        raise HTTPException(status_code=401, detail='invalid API key')

def _language(value, text):
    value = str(value or 'English').strip().lower()
    if value in {'en', 'english'}:
        return 'English'
    if value in {'ja', 'jp', 'japanese'}:
        return 'Japanese'
    if value == 'auto':
        return 'Japanese' if any('\u3040' <= char <= '\u30ff' or '\u4e00' <= char <= '\u9fff' for char in text) else 'English'
    raise HTTPException(status_code=422, detail='language must be Auto, English, or Japanese')

class TtsRequest(BaseModel):
    text: str = Field(min_length=1, max_length=MAX_TEXT_CHARS)
    language: str = Field(default='English', max_length=16)

def _generate(text, language):
    started = time.perf_counter()
    budget = min(MAX_NEW_TOKENS, max(128, len(text) * 3))
    with torch.inference_mode():
        wavs, sample_rate = model.generate_voice_clone(
            text=text.strip(),
            language=language,
            voice_clone_prompt=voice_clone_prompt,
            max_new_tokens=budget,
        )
    samples = np.asarray(wavs[0], dtype=np.float32).reshape(-1)
    samples = np.nan_to_num(samples, nan=0.0, posinf=0.0, neginf=0.0)
    if samples.size == 0 or float(np.sqrt(np.mean(samples * samples))) < 1e-4:
        raise RuntimeError('Qwen returned near-silent audio')
    output = io.BytesIO()
    sf.write(output, np.clip(samples, -1.0, 1.0), int(sample_rate), format='WAV', subtype='PCM_16')
    return output.getvalue(), int(sample_rate), time.perf_counter() - started

@app.get('/health')
async def health(x_api_key: str | None = Header(default=None, alias='X-Api-Key')):
    _authorize(x_api_key)
    return {
        'status': 'ready',
        'mode': 'trained-base-full-icl',
        'profile': 'epoch-0',
        'model': MODEL_ID,
        'base_model': BASE_MODEL_ID,
        'checkpoint_id': 'epoch-0',
        'checkpoint_id_source': f'huggingface:{HF_REVISION}',
        'generation_version': GENERATION_VERSION,
        'deployment_stage': 'isolated',
        'approval_status': 'APPROVED_FOR_ISOLATED_BASE_FULL_ICL',
        'reference_required': True,
        'runtime_lora_loaded': False,
        'conditioning': {
            'reference_required': True,
            'reference_sha256': sha256_file(REFERENCE_AUDIO),
            'reference_transcript_sha256': sha256_file(REFERENCE_TEXT),
            'x_vector_only_mode': False,
        },
        'queue_busy': MODEL_LOCK.locked(),
    }

@app.post('/tts')
async def tts(request: TtsRequest, x_api_key: str | None = Header(default=None, alias='X-Api-Key')):
    _authorize(x_api_key)
    language = _language(request.language, request.text)
    async with MODEL_LOCK:
        audio, sample_rate, elapsed = await asyncio.to_thread(_generate, request.text, language)
    return Response(
        content=audio, media_type='audio/wav',
        headers={
            'X-Tsuki-Backend': 'colab-qwen3-tts-epoch0-base-full-icl',
            'X-Tsuki-Sample-Rate': str(sample_rate),
            'X-Tsuki-Generation-Ms': str(round(elapsed * 1000)),
        },
    )

@app.get('/')
async def root():
    return JSONResponse({'service': 'tsuki-epoch0-colab-tts', 'health': '/health', 'tts': '/tts'})

print('API app created; mode=trained-base-full-icl')

In [ ]:
import uvicorn

uvicorn_config = uvicorn.Config(app, host='127.0.0.1', port=8100, log_level='warning')
uvicorn_server = uvicorn.Server(uvicorn_config)
uvicorn_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
uvicorn_thread.start()
time.sleep(2)
print('Local TTS server running:', uvicorn_thread.is_alive())

In [ ]:
import requests

health_response = requests.get('http://127.0.0.1:8100/health', headers={'X-Api-Key': API_KEY}, timeout=30)
print('local_health_status=', health_response.status_code)
print(json.dumps(health_response.json(), ensure_ascii=False, indent=2))
smoke_response = requests.post(
    'http://127.0.0.1:8100/tts',
    headers={'X-Api-Key': API_KEY},
    json={'text': 'Hello, I am happy to talk with you tonight.', 'language': 'English'},
    timeout=180,
)
smoke_response.raise_for_status()
Path('/content/epoch0_colab_smoke.wav').write_bytes(smoke_response.content)
print('local_tts_status=', smoke_response.status_code, 'bytes=', len(smoke_response.content))

In [ ]:
import stat
import subprocess
import urllib.request

CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CLOUDFLARED,
    )
    CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IXUSR)

_cf_credentials_input = Path('/content/tsuki_colab_tunnel_credentials.json')
if _cf_credentials_input.is_file():
    _cf_credentials = json.loads(_cf_credentials_input.read_text(encoding='utf-8'))
    CF_TUNNEL_ID = str(_cf_credentials.get('TunnelID') or _cf_credentials.get('tunnel_id') or '').strip()
    if not CF_TUNNEL_ID:
        raise RuntimeError('Cloudflare credentials are missing TunnelID')
    CF_CREDENTIAL_FILE = Path('/content/cloudflared-tunnel-credentials.json')
    CF_CREDENTIAL_FILE.write_text(json.dumps(_cf_credentials), encoding='utf-8')
    CF_CREDENTIAL_FILE.chmod(0o600)
    _cf_credentials_input.unlink()
    CF_CONFIG = Path('/content/cloudflared-config.yml')
    CF_CONFIG.write_text(
        f'tunnel: {CF_TUNNEL_ID}\n'
        f'credentials-file: {CF_CREDENTIAL_FILE}\n'
        'ingress:\n'
        '  - hostname: colab-tts.tsukihime.dpdns.org\n'
        '    service: http://127.0.0.1:8100\n'
        '  - service: http_status:404\n',
        encoding='utf-8',
    )
    CF_CONFIG.chmod(0o600)
    _cloudflared_args = [str(CLOUDFLARED), 'tunnel', '--no-autoupdate', '--config', str(CF_CONFIG), 'run', CF_TUNNEL_ID]
else:
    _cf_token_file = Path('/content/tsuki_cloudflare_tunnel_token')
    if _cf_token_file.is_file():
        os.environ['CF_TUNNEL_TOKEN'] = _cf_token_file.read_text(encoding='utf-8').strip()
        _cf_token_file.unlink()
    CF_TUNNEL_TOKEN = os.environ.get('CF_TUNNEL_TOKEN') or getpass.getpass('Cloudflare tunnel token for tsuki-colab-qwen-custom: ')
    if not CF_TUNNEL_TOKEN:
        raise RuntimeError('CF_TUNNEL_TOKEN is required')
    CF_TOKEN_FILE = Path('/content/cloudflared-token')
    CF_TOKEN_FILE.write_text(CF_TUNNEL_TOKEN, encoding='utf-8')
    CF_TOKEN_FILE.chmod(0o600)
    del CF_TUNNEL_TOKEN
    _cloudflared_args = [str(CLOUDFLARED), 'tunnel', '--no-autoupdate', 'run', '--token-file', str(CF_TOKEN_FILE)]

cloudflared_process = subprocess.Popen(
    _cloudflared_args,
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
time.sleep(5)
print('cloudflared_running=', cloudflared_process.poll() is None)

In [ ]:
PUBLIC_TTS_URL = 'https://colab-tts.tsukihime.dpdns.org'
public_health = None
for _attempt in range(12):
    try:
        candidate = requests.get(PUBLIC_TTS_URL + '/health', headers={'X-Api-Key': API_KEY}, timeout=30)
        if candidate.status_code == 200:
            public_health = candidate
            break
        print('public_health_retry=', _attempt + 1, 'status=', candidate.status_code)
    except requests.RequestException as error:
        print('public_health_retry=', _attempt + 1, 'error=', type(error).__name__)
    time.sleep(5)
if public_health is None:
    raise RuntimeError('Public Colab TTS health did not become ready')
print('public_health_status=', public_health.status_code)
print(json.dumps(public_health.json(), ensure_ascii=False, indent=2))

Keep this runtime running while the Discord bot is in use. The Colab runtime is ephemeral; if it stops, the VPS will fail its Qwen health check until this notebook is run again. The API key is passed to the VPS separately and is never printed by this notebook.